In [1]:
# Paste this entire block into a Google Colab cell and run.
# If ipywidgets is not installed, the cell will install and enable it.

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except Exception:
    !pip install ipywidgets
    import ipywidgets as widgets
    from IPython.display import display, clear_output

# --- Safe expression evaluator using ast ---
import ast
import operator as op

# supported operators mapping
_allowed_operators = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.Mod: op.mod,
    ast.USub: op.neg,
    ast.UAdd: op.pos,
}

def safe_eval(expr: str):
    """
    Evaluate arithmetic expressions safely.
    Supports numbers, + - * / % ** and parentheses.
    Raises ValueError for invalid expressions.
    """
    if not isinstance(expr, str):
        raise ValueError("Expression must be a string.")
    expr = expr.strip()
    if expr == "":
        raise ValueError("Empty expression.")

    def _eval(node):
        if isinstance(node, ast.Num):  # <number>
            return node.n
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            left = _eval(node.left)
            right = _eval(node.right)
            op_type = type(node.op)
            if op_type in _allowed_operators:
                return _allowed_operators[op_type](left, right)
            raise ValueError(f"Unsupported binary operator: {op_type}")
        if isinstance(node, ast.UnaryOp):
            operand = _eval(node.operand)
            op_type = type(node.op)
            if op_type in _allowed_operators:
                return _allowed_operators[op_type](operand)
            raise ValueError(f"Unsupported unary operator: {op_type}")
        if isinstance(node, ast.Expr):
            return _eval(node.value)
        raise ValueError(f"Unsupported expression: {type(node)}")

    try:
        parsed = ast.parse(expr, mode='eval')
        return _eval(parsed.body)
    except ZeroDivisionError:
        raise
    except Exception as e:
        raise ValueError("Invalid expression") from e

# --- UI widgets ---
expr_input = widgets.Text(
    value='',
    placeholder='Type an expression (e.g. 12+3*(4-1) / 2)',
    description='Expr:',
    layout=widgets.Layout(width='70%')
)

calc_button = widgets.Button(description='Calculate', button_style='success')
clear_button = widgets.Button(description='Clear', button_style='warning')
history_out = widgets.Output(layout=widgets.Layout(border='1px solid lightgray', height='200px', overflow='auto'))
result_out = widgets.Output(layout=widgets.Layout(border='1px solid lightgray', padding='8px'))

# History storage
history = []

def update_history(expr, result):
    history.append((expr, result))
    with history_out:
        clear_output(wait=True)
        for i, (e, r) in enumerate(reversed(history[-50:]), 1):
            print(f"{len(history)-i+1:03d}. {e}  =>  {r}")

def on_calc_clicked(b):
    expr = expr_input.value
    with result_out:
        clear_output(wait=True)
        try:
            val = safe_eval(expr)
            print(f"Result: {val}")
            update_history(expr, val)
        except ZeroDivisionError:
            print("Error: Division by zero.")
        except Exception as e:
            print("Error:", str(e))

def on_clear_clicked(b):
    expr_input.value = ""
    with result_out:
        clear_output(wait=True)
    with history_out:
        clear_output(wait=True)
    history.clear()

calc_button.on_click(on_calc_clicked)
clear_button.on_click(on_clear_clicked)

# Keyboard "Enter" triggers calculation
def on_submit(change):
    if change['name'] == 'value' and change['new'] != change['old']:
        # do nothing on typing; bind Enter using on_submit handler of Text (works in Jupyter)
        pass

expr_input.on_submit(lambda *_: on_calc_clicked(None))

# Layout
controls = widgets.HBox([calc_button, clear_button])
ui = widgets.VBox([
    widgets.HTML("<h3>Calculator (safe)</h3><small>Supports + - * / % ** and parentheses</small>"),
    expr_input,
    controls,
    widgets.HTML("<b>Output</b>"),
    result_out,
    widgets.HTML("<b>History (most recent at bottom)</b>"),
    history_out
])

display(ui)

# Pre-fill with some examples
examples = ["2+3*4", "10/3", "2**8", "-5 + (3*4)"]
if not history:
    for ex in examples:
        try:
            res = safe_eval(ex)
            update_history(ex, res)
        except Exception:
            pass

/tmp/ipython-input-1229297364.py:41: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  if isinstance(node, ast.Num):  # <number>
/tmp/ipython-input-1229297364.py:42: DeprecationWarning: Attribute n is deprecated and will be removed in Python 3.14; use value instead
  return node.n
